# DifMod — обучение в Google Colab

Запускай ячейки по порядку. В Colab выбери **Среда выполнения → Сменить среду выполнения → T4 GPU** (если доступна). Этот ноутбук обучает нашу архитектуру с нуля; pretrained-веса не скачиваются.

**Данные:** положи папку датасета `DifModDataset` в корень Google Drive. Внутри должен быть `manifest.jsonl` и папка `images/`. Формат каждой строки манифеста: `{"image":"images/example.jpg","caption":"a red car"}`.

Контрольные точки сохраняются на Google Drive, чтобы можно было продолжить после отключения Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch, os, shutil, subprocess, sys

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU не обнаружена. Открой Runtime → Change runtime type и выбери GPU.')

In [ ]:
# Получаем актуальный код DifMod из GitHub
REPO_URL = 'https://github.com/a89526425040-creator/DifMod.git'
REPO_DIR = Path('/content/DifMod')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)

# Colab уже содержит CUDA-сборку PyTorch. Устанавливаем остальные зависимости.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'Pillow>=10.0.0'], check=True)
print('Код загружен:', REPO_DIR)

In [ ]:
# Пути к датасету и сохранениям на Google Drive
DRIVE_DATASET = Path('/content/drive/MyDrive/DifModDataset')
DRIVE_CHECKPOINTS = Path('/content/drive/MyDrive/DifModCheckpoints')
MANIFEST = DRIVE_DATASET / 'manifest.jsonl'
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

if not MANIFEST.is_file():
    raise FileNotFoundError(f'Не найден {MANIFEST}. Создай папку DifModDataset на Google Drive и положи туда manifest.jsonl и изображения.')

# Копируем данные на временный диск Colab: чтение изображений обычно быстрее.
LOCAL_DATASET = Path('/content/DifModDataset')
if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)
LOCAL_MANIFEST = LOCAL_DATASET / 'manifest.jsonl'
num_examples = sum(1 for line in LOCAL_MANIFEST.open(encoding='utf-8') if line.strip())
print('Примеров в датасете:', num_examples)
print('Папка контрольных точек:', DRIVE_CHECKPOINTS)

## Этап 1 — обучение Autoencoder

Для первого пробного запуска используем небольшую batch size и 2 эпохи. После проверки качества реконструкций можно увеличить число эпох. Если Colab отключится, повтори ячейку с `RESUME = True`.

In [ ]:
AE_CHECKPOINT = DRIVE_CHECKPOINTS / 'autoencoder.pt'
AE_EPOCHS = 2          # увеличим после первого успешного теста
BATCH_SIZE = 2         # при нехватке памяти уменьши до 1
IMAGE_SIZE = 256
RESUME = AE_CHECKPOINT.exists()

cmd = [sys.executable, 'train_autoencoder.py',
       '--manifest', str(LOCAL_MANIFEST),
       '--output', str(AE_CHECKPOINT),
       '--epochs', str(AE_EPOCHS),
       '--batch-size', str(BATCH_SIZE),
       '--image-size', str(IMAGE_SIZE),
       '--workers', '2']
if RESUME:
    cmd.append('--resume')
print('Команда:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Autoencoder checkpoint:', AE_CHECKPOINT)

## Этап 2 — обучение генеративной части

Запускай только после успешного завершения этапа 1. Flow checkpoint также сохраняется после каждой эпохи. При повторном запуске ноутбука включи resume, если файл уже существует.

In [ ]:
FLOW_CHECKPOINT = DRIVE_CHECKPOINTS / 'difmod_flow.pt'
FLOW_EPOCHS = 5       # пробный запуск; качество не гарантируется
RESUME_FLOW = FLOW_CHECKPOINT.exists()

cmd = [sys.executable, 'train_flow.py',
       '--manifest', str(LOCAL_MANIFEST),
       '--autoencoder', str(AE_CHECKPOINT),
       '--output', str(FLOW_CHECKPOINT),
       '--epochs', str(FLOW_EPOCHS),
       '--batch-size', str(BATCH_SIZE),
       '--image-size', str(IMAGE_SIZE),
       '--workers', '2']
if RESUME_FLOW:
    cmd.append('--resume')
print('Команда:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Flow checkpoint:', FLOW_CHECKPOINT)

## Важно

- Это первые технические прогоны, а не финальное обучение модели.
- Не закрывай вкладку во время активной эпохи. Контрольная точка пишется после каждой завершённой эпохи.
- Если сессия прервалась, подключи GPU снова, выполни ячейки сверху и повтори нужный этап: ноутбук подхватит checkpoint.
- Бесплатный Colab может ограничивать время и доступность GPU. Не удаляй папку `DifModCheckpoints` на Google Drive.
- Не загружай в публичный GitHub приватный датасет или чужие изображения без разрешения.